In [29]:
# 0. DATA OVERVIEW — data_wip_v5.csv
# Objectifs :
# - Charger le master dataset (CSV séparé par ';')
# - Contrôles rapides : types, NaN, doublons, catégories
# - Générer un dictionnaire de données (docs/data_dictionary.csv)
# - Résumé exécutable pour le README

from pathlib import Path
import pandas as pd

# --- Chemins ---
ROOT = Path("..").resolve()
DATA = ROOT / "data"
DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA / "data_wip_v5.csv"
assert CSV_PATH.exists(), f"Fichier introuvable : {CSV_PATH}"
CSV_PATH

WindowsPath('C:/Users/benoi/Desktop/github/data_science_certification_projects/06_final_project/data/data_wip_v5.csv')

In [30]:
# --- Chargement robuste (CSV FR ; séparateur ';') ---
# On force sep=';' (exports FR/Excel), encodage UTF-8 par défaut.
# Si UTF-8 échoue, on tente latin-1 (rarement utile si sep est correct).
try:
    df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, sep=";", encoding="latin-1")

print(f"Loaded {CSV_PATH.name} -> shape={df.shape}")
display(df.head(3))
display(df.info())

Loaded data_wip_v5.csv -> shape=(672, 38)


,Code_Dpt,Département,Région,année,densité_n-2,densité,pop_globale_n-2,pop_globale,tranche_age_0-24,tranche_age_25-59,...,Total_autres_dechets_n-2,Total_autres_dechets,Déblais_gravats_n-2,Déblais_gravats,Déchets_verts_n-2,Déchets_verts,Encombrants_n-2,Encombrants,Matériaux_recyclables_n-2,Matériaux_recyclables
0,01,Ain,Auvergne-Rhône-Alpes,2009,"99,2","101,8",573868,588857,188068,281744,...,3 792,4 804,35 967,37593,44 066,45013,34 182,34729,23 485,25405
1,02,Aisne,Hauts-de-France,2009,"73,1","73,4",537232,539547,170369,249910,...,770,1 180,23 267,24380,23 295,24539,30 099,31593,9 483,10542
2,03,Allier,Auvergne-Rhône-Alpes,2009,"46,6","46,6",342749,342559,86913,152570,...,1 416,1 867,13 572,14374,17 600,18612,11 129,11932,7 841,8564


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 38 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Code_Dpt                                   672 non-null    object 
 1   Département                                672 non-null    object 
 2   Région                                     672 non-null    object 
 3   année                                      672 non-null    int64  
 4   densité_n-2                                672 non-null    object 
 5   densité                                    672 non-null    object 
 6   pop_globale_n-2                            672 non-null    int64  
 7   pop_globale                                672 non-null    int64  
 8   tranche_age_0-24                           672 non-null    int64  
 9   tranche_age_25-59                          672 non-null    int64  
 10  tranche_age_60+           

None

In [31]:
# --- Contrôles rapides ---
dup_count = int(df.duplicated().sum())
na_counts = df.isna().sum().sort_values(ascending=False)

print("🔎 Lignes dupliquées :", dup_count)
display(na_counts.head(30))

🔎 Lignes dupliquées : 0


nbre_entreprises                 96
nbre_entreprises_agricole        96
nbre_entreprises_industrie       96
nbre_entreprises_service         96
Total_autres_dechets_n-2          0
nb_salaries_secteur_industrie     0
nb_salaries_secteur_service       0
tonnage_dechet_produit_n-2        0
tonnage_dechet_produit            0
Total_autres_dechets              0
Département                       0
Déblais_gravats_n-2               0
Déblais_gravats                   0
Déchets_verts_n-2                 0
Déchets_verts                     0
Encombrants_n-2                   0
Encombrants                       0
Matériaux_recyclables_n-2         0
nb_salaries_secteur_agricole      0
Code_Dpt                          0
csp8_sans_activité                0
csp7_retraités                    0
Région                            0
année                             0
densité_n-2                       0
densité                           0
pop_globale_n-2                   0
pop_globale                 

In [32]:
# --- Typage (aperçu) ---
dtypes_df = (
    pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)})
    .sort_values("dtype")
    .reset_index(drop=True)
)
dtypes_df.head(20)

,column,dtype
0,nbre_entreprises_agricole,float64
1,nbre_entreprises,float64
2,csp8_sans_activité,int64
3,Encombrants,int64
4,Déchets_verts,int64
5,Déblais_gravats,int64
6,nb_salaries_secteur_service,int64
7,nb_salaries_secteur_industrie,int64
8,nb_salaries_secteur_agricole,int64
9,csp7_retraités,int64


In [33]:
# --- Aperçu des catégories courtes (<= 15 valeurs uniques) ---
cat_preview = []
for c in df.columns:
    if df[c].dtype == "object":
        nunq = df[c].nunique(dropna=True)
        if nunq <= 15:
            vals = sorted([str(v) for v in df[c].dropna().unique().tolist()])[:15]
            cat_preview.append({"column": c, "n_unique": nunq, "values": vals})
pd.DataFrame(cat_preview)

,column,n_unique,values
0,Région,13,"[Auvergne-Rhône-Alpes, Bourgogne-Franche-Comté..."


In [34]:
# --- Normalisations légères optionnelles ---
# (Décommente/ajuste si utile pour ton dataset)
# - convertir d’éventuels bytes -> str
def to_text(x):
    if isinstance(x, (bytes, bytearray)):
        try:
            return x.decode("utf-8", "ignore")
        except Exception:
            return str(x)
    return x


obj_cols = [c for c in df.columns if df[c].dtype == "object"]
for c in obj_cols:
    df[c] = df[c].map(to_text)

# Exemple spécifique "Code_Dpt" -> texte zfill(2)
if "Code_Dpt" in df.columns:
    df["Code_Dpt"] = (
        df["Code_Dpt"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(2)
    )

# Exemple dates (adapte la liste)
for cand in ["date", "Date", "DATE"]:
    if cand in df.columns:
        df[cand] = pd.to_datetime(df[cand], errors="coerce")

In [35]:
# --- Dictionnaire de données (auto) ---
data_dict = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum().values,
        "example": [
            df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns
        ],
        "description": ["TODO: décrire la sémantique de la colonne"] * len(df.columns),
    }
)
data_dict_path = DOCS / "data_dictionary.csv"
data_dict.to_csv(data_dict_path, index=False, encoding="utf-8")
data_dict_path

WindowsPath('C:/Users/benoi/Desktop/github/data_science_certification_projects/06_final_project/docs/data_dictionary.csv')

In [36]:
# --- Checks de cohérence simples (à adapter à tes colonnes) ---
issues = []

# Variables "standards" si présentes
COL_YEAR = "annee"
COL_REGION = "region"
COL_TYPE = "typologie"
COL_Y = "tonnage"

# Année plausible
if COL_YEAR in df.columns:
    bad_years = df.loc[~df[COL_YEAR].between(2000, 2100, inclusive="both"), COL_YEAR]
    if not bad_years.empty:
        issues.append(
            f"Valeurs 'annee' hors plage : {sorted(bad_years.dropna().unique().tolist())}"
        )

# Tonnage non négatif
if COL_Y in df.columns and pd.api.types.is_numeric_dtype(df[COL_Y]):
    if (df[COL_Y] < 0).any():
        issues.append("Des valeurs négatives détectées dans 'tonnage'.")

# Région/Typologie non vides (si colonnes présentes)
for col in [COL_REGION, COL_TYPE]:
    if col in df.columns and df[col].isna().all():
        issues.append(f"Toutes les valeurs de '{col}' sont NaN.")

if issues:
    print("⚠️ Problèmes détectés :")
    for e in issues:
        print(" -", e)
else:
    print("Checks de base OK.")

Checks de base OK.


In [37]:
# --- Résumé exécutable (à copier dans le README si besoin) ---
summary = {
    "file": str(CSV_PATH.relative_to(ROOT)),
    "shape": list(df.shape),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "n_duplicates": dup_count,
    "n_cols_with_missing": int((df.isna().sum() > 0).sum()),
    "data_dictionary": str(data_dict_path.relative_to(ROOT)),
    "columns_sample": df.columns[:10].tolist(),
}
summary

{'file': 'data\\data_wip_v5.csv',
 'shape': [672, 38],
 'n_rows': 672,
 'n_columns': 38,
 'n_duplicates': 0,
 'n_cols_with_missing': 4,
 'data_dictionary': 'docs\\data_dictionary.csv',
 'columns_sample': ['Code_Dpt',
  'Département',
  'Région',
  'année',
  'densité_n-2',
  'densité',
  'pop_globale_n-2',
  'pop_globale',
  'tranche_age_0-24',
  'tranche_age_25-59']}